# SageServe 扩缩容分析

分析 Arbiters 日志中的实例扩缩容决策，绘制实例数随时间变化曲线。

数据来源:
- `arbiters/{region}/0.csv`, `1.csv` — 扩缩容决策日志 (timestamp, model, prod, spot, memory_util)
- `memory/0.csv` — GPU 内存快照 (time, instance, memory, max_memory, model_memory, pending_requests)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path
import glob

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

# 结果目录 (相对于 notebook 所在位置)
RESULT_DIR = Path('.')

REGIONS = ['westus', 'centralus', 'eastus']
MODELS = ['A', 'B', 'C', 'D']
REGION_COLORS = {'westus': '#2196F3', 'centralus': '#FF9800', 'eastus': '#4CAF50'}
MODEL_COLORS = {'A': '#E91E63', 'B': '#9C27B0', 'C': '#3F51B5', 'D': '#00BCD4'}

In [ ]:
def load_arbiter_data(result_dir):
    """加载所有区域的 Arbiter CSV 并合并为一个 DataFrame."""
    dfs = []
    for region in REGIONS:
        for csv_file in sorted(glob.glob(str(result_dir / 'arbiters' / region / '*.csv'))):
            df = pd.read_csv(csv_file)
            df['region'] = region
            dfs.append(df)
    if not dfs:
        raise FileNotFoundError(f"未找到 arbiter CSV 文件，请检查路径: {result_dir / 'arbiters'}")
    data = pd.concat(dfs, ignore_index=True)
    data['total'] = data['prod'] + data['spot']
    return data

def load_memory_data(result_dir):
    """加载内存快照 CSV."""
    dfs = []
    for csv_file in sorted(glob.glob(str(result_dir / 'memory' / '*.csv'))):
        df = pd.read_csv(csv_file)
        dfs.append(df)
    if not dfs:
        raise FileNotFoundError(f"未找到 memory CSV 文件，请检查路径: {result_dir / 'memory'}")
    data = pd.concat(dfs, ignore_index=True)
    # 解析 instance 名称: {region}_{model}_{app_id}_{instance_id}
    parts = data['instance'].str.extract(r'^(\w+)_([A-D])_(\d+)_(\d+)$')
    data['region'] = parts[0]
    data['model'] = parts[1]
    data['app_id'] = parts[2].astype(int)
    data['inst_id'] = parts[3].astype(int)
    # 转换内存为 GB
    data['memory_gb'] = data['memory'] / (1024**3)
    data['max_memory_gb'] = data['max_memory'] / (1024**3)
    data['model_memory_gb'] = data['model_memory'] / (1024**3)
    return data

# 加载数据
arbiter_df = load_arbiter_data(RESULT_DIR)
memory_df = load_memory_data(RESULT_DIR)

print(f"Arbiter 数据: {len(arbiter_df)} 行")
print(f"时间范围: {arbiter_df['timestamp'].min():.0f}s ~ {arbiter_df['timestamp'].max():.0f}s")
print(f"Memory 数据: {len(memory_df)} 行")
print(f"\nArbiter 数据样例:")
display(arbiter_df.head(8))
print(f"\nMemory 数据样例:")
display(memory_df.head(8))

## 1. 各区域各模型实例总数随时间变化

`prod + spot` = 该模型当前运行的实例总数

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True, sharey=True)

for i, region in enumerate(REGIONS):
    ax = axes[i]
    region_data = arbiter_df[arbiter_df['region'] == region]
    for model in MODELS:
        model_data = region_data[region_data['model'] == model]
        model_data = model_data.sort_values('timestamp')
        ax.step(model_data['timestamp'], model_data['total'],
                where='post', color=MODEL_COLORS[model],
                label=f'Model {model}', linewidth=1.5, alpha=0.85)
    ax.set_ylabel('实例数', fontsize=11)
    ax.set_title(f'{region.upper()}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, ncol=4)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 25)

axes[-1].set_xlabel('模拟时间 (s)', fontsize=12)
fig.suptitle('各区域实例总数随时间变化 (prod + spot)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2. Prod vs Spot 实例数对比 (按区域汇总)

每个区域一张子图，展示该区域所有模型的 prod 和 spot 实例数堆叠面积图（所有模型合计）。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for i, region in enumerate(REGIONS):
    ax = axes[i]
    region_data = arbiter_df[arbiter_df['region'] == region]

    pivot_prod = region_data.pivot_table(
        values='prod', index='timestamp', columns='model', aggfunc='first'
    ).fillna(method='ffill')
    pivot_spot = region_data.pivot_table(
        values='spot', index='timestamp', columns='model', aggfunc='first'
    ).fillna(method='ffill')

    total_prod = pivot_prod.sum(axis=1)
    total_spot = pivot_spot.sum(axis=1)
    total_all = total_prod + total_spot

    ax.fill_between(pivot_prod.index, 0, total_prod, alpha=0.4, color='#2196F3', label='Prod')
    ax.fill_between(pivot_prod.index, total_prod, total_all, alpha=0.4, color='#FF5722', label='Spot')
    ax.step(pivot_prod.index, total_prod, where='post', color='#1565C0', linewidth=1.5)
    ax.step(pivot_prod.index, total_all, where='post', color='#BF360C', linewidth=1.5)

    ax.set_ylabel('实例数', fontsize=11)
    ax.set_title(f'{region.upper()} — Prod vs Spot (全模型合计)', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('模拟时间 (s)', fontsize=12)
fig.suptitle('Prod vs Spot 实例数变化 (按区域汇总)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. 每个 Region 各模型 Prod/Spot 明细

3 行 (区域) × 4 列 (模型) 网格，每个单元格展示该 (区域, 模型) 组合的 prod/spot 堆叠面积图，可直观看到每个模型独立 prod↔spot 互换的时机和幅度。

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(22, 12), sharex=True, sharey=True)

for ri, region in enumerate(REGIONS):
    region_data = arbiter_df[arbiter_df['region'] == region]
    for mi, model in enumerate(MODELS):
        ax = axes[ri, mi]
        subset = region_data[region_data['model'] == model].sort_values('timestamp')

        if len(subset) > 0:
            ax.fill_between(subset['timestamp'], 0, subset['prod'],
                            alpha=0.5, color='#2196F3', step='post', label='Prod')
            ax.fill_between(subset['timestamp'], subset['prod'],
                            subset['prod'] + subset['spot'],
                            alpha=0.5, color='#FF5722', step='post', label='Spot')
            ax.step(subset['timestamp'], subset['prod'],
                    where='post', color='#1565C0', linewidth=1.0)
            ax.step(subset['timestamp'], subset['total'],
                    where='post', color='#BF360C', linewidth=1.0)

        # 列标题: 模型名
        if ri == 0:
            ax.set_title(f'Model {model}', fontsize=13, fontweight='bold',
                         color=MODEL_COLORS[model])
        # 行标签: 区域名
        if mi == 0:
            ax.set_ylabel(f'{region.upper()}\n实例数', fontsize=11, fontweight='bold')
        if ri == 2:
            ax.set_xlabel('时间 (s)', fontsize=9)

        ax.grid(True, alpha=0.25)
        ax.set_ylim(0, 22)

        # 只在第一个子图显示图例
        if ri == 0 and mi == 0:
            ax.legend(loc='upper right', fontsize=8)

fig.suptitle('各 Region 各模型 Prod/Spot 明细', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. 单模型跨区域对比

每个模型一张子图，对比 3 个区域的实例总数变化曲线。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, model in enumerate(MODELS):
    ax = axes[i]
    model_data = arbiter_df[arbiter_df['model'] == model]
    for region in REGIONS:
        region_data = model_data[model_data['region'] == region].sort_values('timestamp')
        ax.step(region_data['timestamp'], region_data['total'],
                where='post', color=REGION_COLORS[region],
                label=region, linewidth=1.5, alpha=0.85)
    ax.set_title(f'Model {model}', fontsize=13, fontweight='bold')
    ax.set_ylabel('实例数', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 25)

axes[2].set_xlabel('模拟时间 (s)', fontsize=12)
axes[3].set_xlabel('模拟时间 (s)', fontsize=12)
fig.suptitle('各模型实例总数 — 跨区域对比', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. 内存利用率随时间变化

Arbiter 记录的 `memory_util` 字段反映 GPU 内存压力，是扩缩容决策的重要依据。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

for i, region in enumerate(REGIONS):
    ax = axes[i]
    region_data = arbiter_df[arbiter_df['region'] == region]
    for model in MODELS:
        model_data = region_data[region_data['model'] == model].sort_values('timestamp')
        ax.plot(model_data['timestamp'], model_data['memory_util'] * 100,
                color=MODEL_COLORS[model], label=f'Model {model}',
                linewidth=1.0, alpha=0.7, marker='.', markersize=1)
    ax.set_ylabel('Memory Util (%)', fontsize=11)
    ax.set_title(f'{region.upper()}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9, ncol=4)
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

axes[-1].set_xlabel('模拟时间 (s)', fontsize=12)
fig.suptitle('GPU 内存利用率随时间变化', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. 实例级别 GPU 内存使用

从 `memory/0.csv` 中抽取有 pending_requests 的活跃实例，观察其显存使用情况。

In [ ]:
# 筛选有 pending_requests 的活跃实例
active_memory = memory_df[memory_df['pending_requests'] > 0]
print(f"活跃记录数: {len(active_memory)} / {len(memory_df)}")

# 选取前 6 个活跃实例进行绘制
top_instances = (active_memory.groupby('instance')['pending_requests']
                 .max().nlargest(6).index.tolist())

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# 上图: 显存使用
ax = axes[0]
for inst in top_instances:
    inst_data = memory_df[memory_df['instance'] == inst].sort_values('time')
    ax.plot(inst_data['time'], inst_data['memory_gb'],
            label=inst, linewidth=1.2, alpha=0.8, marker='.', markersize=0.5)
ax.set_ylabel('GPU Memory (GB)', fontsize=11)
ax.set_title('活跃实例 GPU 显存使用', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# 下图: 等待请求数
ax = axes[1]
for inst in top_instances:
    inst_data = memory_df[memory_df['instance'] == inst].sort_values('time')
    ax.plot(inst_data['time'], inst_data['pending_requests'],
            label=inst, linewidth=1.2, alpha=0.8, marker='.', markersize=0.5)
ax.set_ylabel('Pending Requests', fontsize=11)
ax.set_title('活跃实例等待请求队列长度', fontsize=13, fontweight='bold')
ax.set_xlabel('模拟时间 (s)', fontsize=12)
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 扩缩容事件瀑布图 (按区域/模型)

用热力图展示每个 (区域, 模型) 组合的实例数随时间变化的全景视图。

In [ ]:
# 构建统一时间轴上的实例数矩阵
time_bins = np.arange(0, arbiter_df['timestamp'].max() + 10, 10)

combinations = [f"{r}_{m}" for r in REGIONS for m in MODELS]
heatmap_data = np.zeros((len(combinations), len(time_bins) - 1))

for ci, combo in enumerate(combinations):
    region, model = combo.split('_')
    subset = arbiter_df[(arbiter_df['region'] == region) & (arbiter_df['model'] == model)]
    subset = subset.sort_values('timestamp')
    if len(subset) == 0:
        continue
    for ti in range(len(time_bins) - 1):
        t = time_bins[ti]
        before = subset[subset['timestamp'] <= t]
        if len(before) > 0:
            heatmap_data[ci, ti] = before['total'].iloc[-1]
        elif ti > 0:
            heatmap_data[ci, ti] = heatmap_data[ci, ti - 1]

fig, ax = plt.subplots(figsize=(18, 8))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd',
               extent=[0, time_bins[-2], len(combinations) - 0.5, -0.5])

ax.set_yticks(range(len(combinations)))
ax.set_yticklabels(combinations)
ax.set_xlabel('模拟时间 (s)', fontsize=12)
ax.set_title('实例数全景热力图 (region_model)', fontsize=15, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('实例总数 (prod + spot)', fontsize=11)

for i in range(1, 3):
    ax.axhline(y=i * 4 - 0.5, color='white', linewidth=2, linestyle='-')

for i, region in enumerate(REGIONS):
    ax.text(-50, i * 4 + 1.5, region.upper(), fontsize=11, fontweight='bold',
            va='center', ha='right', color='#333')

plt.tight_layout()
plt.show()

## 8. 扩缩容关键统计

汇总各区域各模型的扩缩容事件次数和实例数变化范围。

> 注: 本模拟场景中实例 **总数 (prod+spot)** 保持恒定 80 (20 实例/模型)，扩缩容表现为 prod 与 spot 之间的互换，而非总数的增减。

In [ ]:
# 检测 prod/spot 变化事件 (本场景中 total 恒定为 20，但 prod/spot 比例在变化)
events = []
for (region, model), group in arbiter_df.groupby(['region', 'model']):
    group = group.sort_values('timestamp')
    prev = None
    for _, row in group.iterrows():
        if prev is not None:
            if row['prod'] != prev['prod'] or row['spot'] != prev['spot']:
                events.append({
                    'region': region,
                    'model': model,
                    'timestamp': row['timestamp'],
                    'prod_from': int(prev['prod']),
                    'prod_to': int(row['prod']),
                    'spot_from': int(prev['spot']),
                    'spot_to': int(row['spot']),
                    'total': int(row['prod'] + row['spot']),
                    'memory_util': row['memory_util']
                })
        prev = {'prod': row['prod'], 'spot': row['spot']}

if events:
    events_df = pd.DataFrame(events)
    print(f"总 Prod/Spot 变动事件数: {len(events_df)}")
    scale_up = events_df[events_df['prod_to'] > events_df['prod_from']]
    scale_down = events_df[events_df['prod_to'] < events_df['prod_from']]
    print(f"  Prod↑ Spot↓ (扩 prod): {len(scale_up)}")
    print(f"  Prod↓ Spot↑ (缩 prod): {len(scale_down)}")
    print(f"\n变动事件样例 (前 10 条):")
    display(events_df.head(10))
else:
    print("未检测到 Prod/Spot 变动事件")

# 按区域和模型汇总
summary_stats = arbiter_df.groupby(['region', 'model']).agg(
    total_instances=('total', 'first'),
    min_prod=('prod', 'min'),
    max_prod=('prod', 'max'),
    final_prod=('prod', 'last'),
    min_spot=('spot', 'min'),
    max_spot=('spot', 'max'),
    final_spot=('spot', 'last'),
    min_memory_util=('memory_util', 'min'),
    max_memory_util=('memory_util', 'max'),
    mean_memory_util=('memory_util', 'mean'),
).round(2)

print(f"\n各区域各模型实例汇总:")
display(summary_stats)

---
## 总结

- **Total 恒定**: 本模拟中实例总数恒定为 80 (各模型各区域 20 实例上限)，扩缩容表现为 prod ↔ spot 互换
- **Prod vs Spot**: `scaling_level=2` 模式下，Arbiter 根据内存利用率在 prod 和 spot 实例之间动态调整
- **区域差异**: 各区域的扩缩容行为独立，受负载分布影响
- **模型权重**: 模型 A/C（较重）的实例数变化与 B/D（较轻）的差异反映了差异化调度策略
- **Spot 实例**: 随着时间推移，spot 实例数逐渐增加，反映了 Arbiter 在面对内存压力时将部分 prod 实例转为 spot